In [1]:
from IPython.display import display, Markdown

def add_section_title(title, color="black", size="24px"):
    display(Markdown(f"<h2 style='color:{color}; font-size:{size};'>{title}</h2>"))



In [2]:
add_section_title("Section 1: Setup AOAI client for Agent", color="blue", size="18px")

<h2 style='color:blue; font-size:18px;'>Section 1: Setup AOAI client for Agent</h2>

In [3]:
from agent_framework.azure import AzureOpenAIChatClient
import os

client = AzureOpenAIChatClient(
    endpoint=os.environ["AOAI_API_BASE"],
    api_key=os.environ["AOAI_API_KEY"],
    api_version=os.environ["AOAI_LLM_API_VERSION"],
    deployment_name=os.environ["AOAI_LLM_DEPLOYMENT"],
)


In [4]:
from agent_framework import Agent

In [5]:
add_section_title("Section 2: Multiple ways to check MCP tools", color="blue", size="18px")

<h2 style='color:blue; font-size:18px;'>Section 2: Multiple ways to check MCP tools</h2>

In [6]:
add_section_title("Section 2.1: Display all tools as one unified MCP", color="blue", size="18px")

<h2 style='color:blue; font-size:18px;'>Section 2.1: Display all tools as one unified MCP</h2>

In [7]:
from agent_framework import MCPStreamableHTTPTool

mcp = MCPStreamableHTTPTool(
    name="Unified MCP",
    url="http://127.0.0.1:8000/mcp",
    approval_mode="never_require",
)

async with mcp:
    # show what tools were loaded from the server
    print("Loaded tools:", [f.name for f in mcp.functions])

Loaded tools: ['graphrag_search', 'age_get_schema_cached', 'age_cypher_query', 'age_entity_lookup', 'age_nl2cypher_query']


In [8]:
add_section_title("Section 2.2: Separate graphrag_search and age_tool", color="blue", size="18px")

<h2 style='color:blue; font-size:18px;'>Section 2.2: Separate graphrag_search and age_tool</h2>

In [9]:
from agent_framework import MCPStreamableHTTPTool

MCP_URL = "http://127.0.0.1:8000/mcp"

# GraphRAG-only MCP wrapper
graphrag_tool = MCPStreamableHTTPTool(
    name="GraphRAG MCP",
    url=MCP_URL,
    approval_mode="never_require",
    allowed_tools=["graphrag_search"],   # ✅ hard restriction
)

# AGE-only MCP wrapper
age_tool = MCPStreamableHTTPTool(
    name="AGE MCP",
    url=MCP_URL,
    approval_mode="never_require",
    allowed_tools=["age_get_schema_cached", "age_cypher_query", "age_entity_lookup", "age_nl2cypher_query"],  # ✅ hard restriction
)

In [10]:
add_section_title("Section 3: graphRAG search using MCP call and agent", color="blue", size="18px")

<h2 style='color:blue; font-size:18px;'>Section 3: graphRAG search using MCP call and agent</h2>

In [11]:
add_section_title("Section 3.1: Setup run-time tune-able parameters", color="blue", size="18px")

<h2 style='color:blue; font-size:18px;'>Section 3.1: Setup run-time tune-able parameters</h2>

In [12]:
local_params = {
    "community_level": 2,
    "response_type": "multiple paragraphs",
}

global_params = {
    "community_level": 2,
    "response_type": "multiple paragraphs",
    "dynamic_community_selection": True,
}



In [14]:
add_section_title("Section 3.2: Local search example using MCP call", color="blue", size="18px")

<h2 style='color:blue; font-size:18px;'>Section 3.2: Local search example using MCP call</h2>

In [15]:
#local search

from agent_framework import MCPStreamableHTTPTool

MCP_URL = "http://127.0.0.1:8000/mcp"

async with MCPStreamableHTTPTool(
    name="Unified MCP",
    url=MCP_URL,
    approval_mode="never_require",
    request_timeout=120,
    sse_read_timeout=300,
    allowed_tools=["graphrag_search"],
) as mcp:

    out = await mcp.call_tool(
        "graphrag_search",
        query="Summarize the main topics in this dataset in 3 bullet points.",
        search_method="local",
        params=local_params,
    )
    
    print(out)


{
  "tool": "graphrag_search",
  "input": {
    "query": "Summarize the main topics in this dataset in 3 bullet points.",
    "search_method": "local",
    "params": {
      "community_level": 2,
      "response_type": "multiple paragraphs"
    }
  },
  "result": "### Summary of Main Topics\n\n- **Technological Leadership and Contributions**: \n   - The dataset highlights notable contributions of industry leaders such as Kevin Scott, Alice Steinglass, Anders Hejlsberg, Glen Weyl, and Jaron Lanier, emphasizing their roles in shaping technology, computer science education, programming languages, and data ethics. Key topics include artificial intelligence, educational advocacy (e.g., Code.org's 'Hour of Code'), and ethical considerations in technology like facial recognition and data dignity [Data: Reports (118, 133, 105, 151); Entities (72, 290, 56); Relationships (489, 1178, 116, 115, +more)].\n\n- **Machine Learning and Artificial Intelligence Innovations**:\n   - The transformational 

In [16]:
add_section_title("Section 3.3: Local search example using agent", color="blue", size="18px")

<h2 style='color:blue; font-size:18px;'>Section 3.3: Local search example using agent</h2>

In [17]:
local_search_agent = Agent(
    client=client,
    tools=[graphrag_tool],
    system_prompt=f"""
You are a GraphRAG agent.

You MUST call the graphrag_search tool.

When calling the tool:
- query = the user question
- search_method = "local"
- params = {local_params}

Rules:
- Do NOT answer from prior knowledge
- Return ONLY the tool result
""",
)

In [18]:
response = await local_search_agent.run(
    "Summarize the main topics in this dataset in 3 bullet points."
)

print(response)

### Key Topics

1. **Influential Individuals and Their Contributions:**
   - Key figures like Kevin Scott, Jaron Lanier, Dio Gonzalez, and Anders Hejlsberg have made significant impacts in areas such as artificial intelligence, ethical technology, mixed reality, and programming innovations.

2. **Technologies Driving Innovation:**
   - Notable advancements include unsupervised learning for natural language processing, and developments in virtual and mixed reality, such as VR facilities and animation tools.

3. **Ethical and Societal Implications of Technology:**
   - Discussions on ethical concerns, including fairness in AI, data ownership, digital economies, and inclusivity, highlight the need for responsible innovation and societal mindfulness.


In [19]:
add_section_title("Section 3.4: Global search example using MCP call", color="blue", size="18px")

<h2 style='color:blue; font-size:18px;'>Section 3.4: Global search example using MCP call</h2>

In [20]:
#global search

from agent_framework import MCPStreamableHTTPTool

MCP_URL = "http://127.0.0.1:8000/mcp"

async with MCPStreamableHTTPTool(
    name="Unified MCP",
    url=MCP_URL,
    approval_mode="never_require",
    request_timeout=120,
    sse_read_timeout=300,
    allowed_tools=["graphrag_search"],
) as mcp:

    
    out = await mcp.call_tool(
        "graphrag_search",
        query="Summarize the main topics in this dataset in 3 bullet points.",
        search_method="global",
        params=global_params,
    )

    print(out)


{
  "tool": "graphrag_search",
  "input": {
    "query": "Summarize the main topics in this dataset in 3 bullet points.",
    "search_method": "global",
    "params": {
      "community_level": 2,
      "response_type": "multiple paragraphs",
      "dynamic_community_selection": true
    }
  },
  "result": "- **Technological Advancements and Societal Impact**: The dataset explores breakthroughs in fields like artificial intelligence, programming languages, blockchain, virtual/mixed reality, and advanced software engineering tools. It delves into applications across industries such as healthcare, education, animation, and digital networks, while also addressing ethical concerns, digital equality, and the socio-economic impacts of emerging technologies like cryptocurrency [Data: Reports (34, 136, 162, 104, 146, 142, 118, 122, +more)].\n\n- **Influential Figures and Organizations**: Significant attention is given to key individuals such as Kevin Scott, Reid Hoffman, Jaron Lanier, Judy Est

In [22]:
add_section_title("Section 3.5: Global search using agent", color="blue", size="18px")

<h2 style='color:blue; font-size:18px;'>Section 3.5: Global search using agent</h2>

In [21]:
global_search_agent = Agent(
    client=client,
    tools=[graphrag_tool],
    system_prompt=f"""
You are a GraphRAG agent.

You MUST call the graphrag_search tool exactly once.

You MUST call it with these exact arguments:
- search_method: "global"
- params: {global_params}

The value of `query` MUST be the user's question verbatim.

Rules:
- Do NOT answer from prior knowledge
- Do NOT rephrase or summarize yourself
- Return ONLY the tool result
""",
)


In [22]:
response = await global_search_agent.run(
    "Summarize the main topics in this dataset in 3 bullet points."
)

print(response)

- **Prominent Figures and Technological Advancements**: The dataset highlights key individuals such as Kevin Scott, Alice Steinglass, Jaron Lanier, and Dio Gonzalez. It focuses on their contributions to artificial intelligence, virtual reality, and computer science education, exploring ethical AI deployment, educational advocacy, and major technologies like mixed reality and animation tools.

- **Ethical and Societal Implications of Technology**: Discussions emphasize the ethical and societal effects of technologies like facial recognition, data dignity, and sustainable digital frameworks. Issues such as privacy, equity, and data value collective bargaining are central, with contributions from experts like Kevin Scott and Jaron Lanier.

- **Applications of Machine Learning and AI**: The dataset examines the transformative value of unsupervised learning in tasks like document summarization and translation. It also discusses large-scale computational challenges, highlighting initiatives 

In [14]:
add_section_title("Section 4: Use age_tool in MCP call and agent", color="blue", size="18px")

<h2 style='color:blue; font-size:18px;'>Section 4: Use age_tool in MCP call and agent</h2>

In [71]:
add_section_title("Section 4.1: Call age_tool in MCP, direct Cypher query", color="blue", size="18px")

<h2 style='color:blue; font-size:18px;'>Section 4.1: Call age_tool in MCP, direct Cypher query</h2>

In [10]:
from agent_framework import MCPStreamableHTTPTool

MCP_URL = "http://127.0.0.1:8000/mcp"

cypher = """
MATCH (kevin:Entity)
WHERE toLower(kevin.title) CONTAINS 'kevin scott'
MATCH (kevin)-[:RELATED_TO]-(person:Entity)-[r:RELATED_TO]->(company:Entity)
WHERE toLower(r.description) CONTAINS 'computer science'
   OR toLower(r.description) CONTAINS 'education'
   OR toLower(r.description) CONTAINS 'kids'
RETURN {
  mentioned_person: person.title,
  organization: company.title,
  relationship_description: r.description
} AS result
LIMIT 5
"""

async with MCPStreamableHTTPTool(
    name="Unified MCP",
    url=MCP_URL,
    approval_mode="never_require",
    request_timeout=120,
    sse_read_timeout=300,
    # ✅ allow all tools you intend to call in THIS session
    allowed_tools=["age_get_schema_cached", "age_cypher_query"],
) as mcp:

    # 1) schema cache
    schema = await mcp.call_tool("age_get_schema_cached", refresh=False)
    print("schema cached:", schema)

    # 2) run cypher
    out = await mcp.call_tool("age_cypher_query", cypher=cypher)
    print("cypher result:", out)

schema cached: {
  "tool": "age_get_schema_cached",
  "refresh": true,
  "cached": false,
  "ttl_seconds": 3600,
  "result": {
    "graph": "graphRAG",
    "vertex_labels": [
      "\"Document\"",
      "\"Entity\""
    ],
    "edge_labels": [
      "\"RELATED_TO\""
    ]
  }
}
cypher result: {
  "tool": "age_cypher_query",
  "input": {
    "cypher": "\nMATCH (kevin:Entity)\nWHERE toLower(kevin.title) CONTAINS 'kevin scott'\nMATCH (kevin)-[:RELATED_TO]-(person:Entity)-[r:RELATED_TO]->(company:Entity)\nWHERE toLower(r.description) CONTAINS 'computer science'\n   OR toLower(r.description) CONTAINS 'education'\n   OR toLower(r.description) CONTAINS 'kids'\nRETURN {\n  mentioned_person: person.title,\n  organization: company.title,\n  relationship_description: r.description\n} AS result\nLIMIT 5\n"
  },
  "row_count": 5,
  "rows": [
    {
      "result": "{\"organization\": \"KEVIN SCOTT\", \"mentioned_person\": \"JARON LANIER\", \"relationship_description\": \"Jaron Lanier and Kevin Scott

In [2]:
add_section_title("Section 4.2: Use age_tool in agent, query with natural language, tool converts NL2Cypher.", color="blue", size="18px")

<h2 style='color:blue; font-size:18px;'>Section 4.2: Use age_tool in agent, query with natural language, tool converts NL2Cypher.</h2>

In [10]:
AGE_AGENT_PROMPT = """
You are an AGE graph assistant.

Your job is to answer questions ONLY using the AGE graph via tools.
Do NOT answer from general knowledge.

==============================
STEP 1: Understand the question
==============================

First, classify the question:

A) SIMPLE / DIRECT
- Identity or lookup questions
- Examples:
  - "Who is Alice Steinglass?"
  - "What is Kevin Scott?"
  - "Find entity named X"

B) COMPLEX / REASONING
- Requires interpretation, decomposition, or multi-hop reasoning
- Examples:
  - "Who did Kevin Scott mention?"
  - "Who did X mention that is connected to Y?"
  - "Which people are indirectly related to Z through education?"

==============================
STEP 2: Choose the correct tool
==============================

If the question is SIMPLE / DIRECT:
- Prefer deterministic tools:
  - age_entity_lookup
  - age_cypher_query (simple pattern)

If the question is COMPLEX / REASONING:
- Use:
  - age_nl2cypher_query(question, limit)

IMPORTANT:
- Use EXACTLY ONE tool per question.
- Do NOT call multiple tools.
- Do NOT answer without calling a tool.

==============================
STEP 3: Tool usage rules
==============================

- When calling a tool, pass the ORIGINAL user question.
- Do NOT rewrite or summarize the question unless required by the tool.
- Do NOT add explanations outside the tool result.

==============================
STEP 4: Final output rules
==============================

- If the tool result contains row_count == 0:
  output EXACTLY:
  No matching data found in the graph

- If row_count > 0:
  output ONLY the rows field from the tool result.

- Do NOT add explanations, apologies, or suggestions.
- Stop immediately after output.
"""


In [11]:
add_section_title("Example 1: A simple question.", color="blue", size="18px")

<h2 style='color:blue; font-size:18px;'>Example 1: A simple question.</h2>

In [12]:
age_tool = MCPStreamableHTTPTool(
    name="AGE MCP",
    url=MCP_URL,
    approval_mode="never_require",
    allowed_tools=["age_entity_lookup"],
)

question = (
  "Who is Alice Steinglass?"
)

async with age_tool:
    age_agent = Agent(
        client=client,
        tools=[age_tool],
        system_prompt=AGE_AGENT_PROMPT,
        max_iterations=3,
    )

    resp = await age_agent.run(question)
    print(resp.text)


Alice Steinglass is a prominent leader and advocate in the field of computer science education. She currently serves as the President of Code.org, a nonprofit organization focused on advancing access to computer science education for K-12 students, promoting diversity in technology, and inspiring careers in STEM fields. Under her leadership, Code.org has developed educational curricula, resources, and tools to teach introductory computer science. The organization is also known for the globally recognized "Hour of Code" initiative, which has reached millions of students in over 180 countries.

Alice is a strong advocate for equitable and inclusive education, ensuring that every child, regardless of their background, has the opportunity to learn computer science. She works to remove gender stereotypes, encouraging young girls to engage in technology and coding. Alice also emphasizes empowering teachers and developing innovative teaching methods—even non-computer-based ones—to make comput

In [13]:
def tool_trace(resp):
    traces = []
    for m in (resp.messages or []):
        for c in getattr(m, "contents", []) or []:
            name = getattr(c, "name", None)
            args = getattr(c, "arguments", None)
            result = getattr(c, "result", None)
            if name or args or result:
                traces.append({
                    "content_type": type(c).__name__,
                    "name": name,
                    "arguments": args,
                    "result": result,
                })
    return traces

print("=== TOOL TRACE ===")
for t in tool_trace(resp):
    print(t)


=== TOOL TRACE ===
{'content_type': 'Content', 'name': 'age_entity_lookup', 'arguments': '{"name":"Alice Steinglass","limit":1}', 'result': None}
{'content_type': 'Content', 'name': None, 'arguments': None, 'result': '{\n  "tool": "age_entity_lookup",\n  "input": {\n    "name": "Alice Steinglass",\n    "limit": 1\n  },\n  "row_count": 1,\n  "rows": [\n    {\n      "result": "{\\"type\\": \\"PERSON\\", \\"title\\": \\"ALICE STEINGLASS\\", \\"description\\": \\"Alice Steinglass is a prominent leader and advocate in the field of computer science education, currently serving as the President of Code.org. Code.org is a nonprofit organization dedicated to advancing access to computer science education for K-12 students, promoting diversity in technology, and encouraging careers in STEM fields. Under her leadership, Code.org has developed curricula, tools, and software to support introductory computer science education and has spearheaded initiatives like the globally recognized \\\\\\"Hour o

In [14]:
add_section_title("Example 2: Decompose a complicated question.", color="blue", size="18px")

<h2 style='color:blue; font-size:18px;'>Example 2: Decompose a complicated question.</h2>

In [19]:
age_tool = MCPStreamableHTTPTool(
    name="AGE MCP",
    url=MCP_URL,
    approval_mode="never_require",
    allowed_tools=["age_nl2cypher_query"],
)

question = (
  "There are two persons involved."
  "Source person: Kevin Scott. "
  "Target person: another person mentioned by Kevin Scott."
  "The target person played a pivotal role in advancing the company\'s efforts in virtual and mixed reality."
  "Question: Who did Kevin Scott mention?"
)

async with age_tool:
    age_agent = Agent(
        client=client,
        tools=[age_tool],
        system_prompt=AGE_AGENT_PROMPT,
        max_iterations=3,
    )

    resp = await age_agent.run(question)
    print(resp.text)


Kevin Scott mentioned Dio Gonzalez as the person who played a pivotal role in advancing the company's efforts in virtual and mixed reality. 

From the provided information:
- Dio Gonzalez co-founded the Mixed Reality Research Group at Unity and played a pivotal role in advancing Unity's efforts in virtual and mixed reality. He contributed to developing cutting-edge virtual reality tools and engaged in experimental projects that reshaped how development tools are utilized worldwide.
- Kevin Scott and Dio Gonzalez also discussed their shared experiences and interests in mixed reality, including its applications, during an episode of the *Behind the Tech* podcast.


In [20]:
def tool_trace(resp):
    traces = []
    for m in (resp.messages or []):
        for c in getattr(m, "contents", []) or []:
            name = getattr(c, "name", None)
            args = getattr(c, "arguments", None)
            result = getattr(c, "result", None)
            if name or args or result:
                traces.append({
                    "content_type": type(c).__name__,
                    "name": name,
                    "arguments": args,
                    "result": result,
                })
    return traces

print("=== TOOL TRACE ===")
for t in tool_trace(resp):
    print(t)


=== TOOL TRACE ===
{'content_type': 'Content', 'name': 'age_nl2cypher_query', 'arguments': '{"question":"Who did Kevin Scott mention that played a pivotal role in advancing the company\'s efforts in virtual and mixed reality?","limit":5}', 'result': None}
{'content_type': 'Content', 'name': None, 'arguments': None, 'result': '{\n  "tool": "age_nl2cypher_query",\n  "input": {\n    "question": "Who did Kevin Scott mention that played a pivotal role in advancing the company\'s efforts in virtual and mixed reality?",\n    "limit": 5\n  },\n  "generated_cypher": "MATCH (source:Entity)-[r:RELATED_TO]-(mid:Entity)-[r2:RELATED_TO]-(target:Entity)\\nWHERE toLower(source.title) CONTAINS \'kevin scott\'\\nAND (\\n     toLower(r.description) CONTAINS \'mention\'\\n  OR toLower(r2.description) CONTAINS \'mention\'\\n)\\nAND (\\n     toLower(r.description) CONTAINS \'pivotal role\'\\n  OR toLower(r2.description) CONTAINS \'pivotal role\'\\n)\\nAND (\\n     toLower(r.description) CONTAINS \'virtual a

In [26]:
add_section_title("Section 5: Agent with unified MCP, with router prompt route to the right MCP tool", color="blue", size="18px")

<h2 style='color:blue; font-size:18px;'>Section 5: Agent with unified MCP, with router prompt route to the right MCP tool</h2>

In [10]:
ROUTER_PROMPT = """
You are a tool-first routing agent.

You have access to the following tools:
- graphrag_search(query: str, search_method: str = "local", params: dict | None = None)
- age_entity_lookup(name: str, limit: int = 5)
- age_cypher_query(cypher: str)
- age_nl2cypher_query(question: str, limit: int = 5)
- age_get_schema_cached(refresh: bool = False)

Your job is to route the user request to the most appropriate tool
and return the result.

==============================
Routing Rules
==============================

1) Use graphrag_search IF the user asks for:
   - dataset-wide summaries
   - main themes or topics
   - trends or patterns
   - high-level overviews
   - bullet-point summaries

   Examples:
   - "Summarize the main topics in this dataset"
   - "What are the key themes discussed?"

   Use:
   - graphrag_search(query=<user question>)
   - Use default search_method unless explicitly requested otherwise

2) Use AGE tools IF the user asks for:
   - a specific person or entity
   - relationships between entities
   - graph facts or connections
   - questions starting with "Who", "What is X", "How is A related to B"

   Choose the AGE tool as follows:

   a) Simple identity or lookup questions:
      - Use age_entity_lookup(name, limit)

      Examples:
      - "Who is Alice Steinglass?"
      - "Find entity named Kevin Scott"

   b) Deterministic graph questions with a clear structure:
      - Generate Cypher directly
      - Use age_cypher_query(cypher)

      Examples:
      - "How is A related to B?"
      - "What entities are connected to Kevin Scott?"

   c) Complex or semantic graph questions that:
      - involve multiple entities,
      - use language like "mentioned", "related to", or "connected to",
      - or combine people, topics, and organizations,

      Then:
      - Use age_nl2cypher_query
      - Pass the original user question
      - Use limit <= 5

      Examples:
      - "Who did Kevin Scott mention?"
      - "Who did Kevin Scott mention that is related to computer science or education?"
      - "Who did Kevin Scott mention in relation to virtual or mixed reality?"

==============================
Tool Usage Rules
==============================

- Use EXACTLY ONE tool per user question.
- Do NOT answer from prior knowledge.
- Do NOT call multiple tools.
- Do NOT call age_get_schema_cached unless schema is required to construct Cypher.

- For age_cypher_query:
  - Always include LIMIT <= 5 unless the user explicitly asks for more.

==============================
Failure Handling
==============================

- If a tool returns no rows, respond with:
  "No matching data found in the graph."

==============================
Response Format
==============================

When responding:
- Start with: TOOL USED: <tool_name>
- Then return the tool result exactly as returned.
- Do NOT add explanations or extra text.
"""

In [11]:
from agent_framework import MCPStreamableHTTPTool

MCP_URL = "http://127.0.0.1:8000/mcp"

unified_tool = MCPStreamableHTTPTool(
    name="Unified MCP",
    url=MCP_URL,
    approval_mode="never_require",
    request_timeout=120,
    sse_read_timeout=300,
    allowed_tools=[
        "graphrag_search",
        "age_get_schema_cached",
        "age_entity_lookup",
        "age_cypher_query",
        "age_nl2cypher_query"
    ],
)


In [12]:
def tool_trace_current(response):
    traces = []

    for m in response.messages or []:
        for c in getattr(m, "contents", []) or []:
            # Tool calls / tool results always have a name or a result
            if getattr(c, "name", None) or getattr(c, "result", None):
                traces.append({
                    "content_type": type(c).__name__,
                    "name": getattr(c, "name", None),
                    "arguments": getattr(c, "arguments", None),
                    "result": getattr(c, "result", None),
                })

    return traces

In [13]:
from agent_framework import Agent

router_agent = Agent(
    client=client,          # your AzureOpenAIChatClient
    tools=[unified_tool],   # ✅ pass the MCP wrapper
    system_prompt=ROUTER_PROMPT,
)

In [14]:
add_section_title("Example 1: Routing to age_entity_lookup, age_nl2cypher_query", color="blue", size="18px")

<h2 style='color:blue; font-size:18px;'>Example 1: Routing to age_entity_lookup, age_nl2cypher_query</h2>

In [16]:
resp1 = await router_agent.run("What entities are connected to Bill Gates in this dataset?")
print(resp1)


In this dataset, Bill Gates is connected to the following entity:

- **Connected Entity**: Bitcoin
  - **Relationship Description**: Bill Gates is mentioned in a scenario highlighting Bitcoin's resistance to outside influence and immutability.


In [17]:
print("=== TOOL TRACE ===")
for t in tool_trace_current(resp1):
    print(t)


=== TOOL TRACE ===
{'content_type': 'Content', 'name': 'age_entity_lookup', 'arguments': '{"name":"Bill Gates"}', 'result': None}
{'content_type': 'Content', 'name': None, 'arguments': None, 'result': '{\n  "tool": "age_entity_lookup",\n  "input": {\n    "name": "Bill Gates",\n    "limit": 5\n  },\n  "row_count": 1,\n  "rows": [\n    {\n      "result": "{\\"type\\": \\"PERSON\\", \\"title\\": \\"BILL GATES\\", \\"description\\": \\"Bill Gates, the co-founder of Microsoft, is referenced in a scenario showcasing the sovereignty and immutable nature of Bitcoin and blockchain transactions\\"}"\n    }\n  ]\n}'}
{'content_type': 'Content', 'name': 'age_nl2cypher_query', 'arguments': '{"question":"What entities are connected to Bill Gates?"}', 'result': None}
{'content_type': 'Content', 'name': None, 'arguments': None, 'result': '{\n  "tool": "age_nl2cypher_query",\n  "input": {\n    "question": "What entities are connected to Bill Gates?",\n    "limit": 5\n  },\n  "generated_cypher": "MATCH 

In [18]:
add_section_title("Example 2: Routing to age_entity_lookup", color="blue", size="18px")

<h2 style='color:blue; font-size:18px;'>Example 2: Routing to age_entity_lookup</h2>

In [19]:
resp2 = await router_agent.run("Who is Alice Steinglass?")

print(resp2)


Alice Steinglass is a prominent leader and advocate in the field of computer science education. She is currently serving as the President of Code.org, a nonprofit organization dedicated to:

- Expanding access to computer science education for K-12 students.
- Promoting diversity in technology.
- Encouraging careers in STEM fields.

Under her leadership, Code.org has developed curricula, tools, and software to support introductory computer science education. She has also spearheaded initiatives such as the globally recognized "Hour of Code," which has reached millions of students in more than 180 countries.

Alice is a strong advocate for equitable and inclusive education. She strives to ensure that all children, regardless of their background, have access to computer science learning opportunities. She champions initiatives to break gender stereotypes in coding, encouraging early involvement in technology, particularly for young girls. Her focus includes making computer science educat

In [21]:
print("=== TOOL TRACE ===")
for t in tool_trace_current(resp2):
    print(t)


=== TOOL TRACE ===
{'content_type': 'Content', 'name': 'age_entity_lookup', 'arguments': '{"name":"Alice Steinglass"}', 'result': None}
{'content_type': 'Content', 'name': None, 'arguments': None, 'result': '{\n  "tool": "age_entity_lookup",\n  "input": {\n    "name": "Alice Steinglass",\n    "limit": 5\n  },\n  "row_count": 1,\n  "rows": [\n    {\n      "result": "{\\"type\\": \\"PERSON\\", \\"title\\": \\"ALICE STEINGLASS\\", \\"description\\": \\"Alice Steinglass is a prominent leader and advocate in the field of computer science education, currently serving as the President of Code.org. Code.org is a nonprofit organization dedicated to advancing access to computer science education for K-12 students, promoting diversity in technology, and encouraging careers in STEM fields. Under her leadership, Code.org has developed curricula, tools, and software to support introductory computer science education and has spearheaded initiatives like the globally recognized \\\\\\"Hour of Code,\\\

In [34]:
add_section_title("Example 3: Routing to graphrag_search", color="blue", size="18px")

<h2 style='color:blue; font-size:18px;'>Example 3: Routing to graphrag_search</h2>

In [32]:
resp3 = await router_agent.run("What recurring ideas or patterns appear across the documents?")

print(resp3)


The recurring ideas and patterns emerging across the documents involve several interconnected themes that are particularly relevant in the context of technology, innovation, ethics, and interdisciplinary collaboration. Here is an overview of these key themes:

---

### 1. **Innovation as a Driving Force Across Disciplines**
   - Innovation plays a central role in various fields like technology and artificial intelligence.
   - Key examples include:
     - Jaron Lanier's work in virtual reality, transforming human experiences.
     - Reid Hoffman's creation of LinkedIn, revolutionizing professional networking.
     - Judy Estrin’s early contributions to foundational internet technologies.

---

### 2. **Interdisciplinary Collaboration and Mutual Influence**
   - There is a recurring emphasis on how disciplines like art and technology enhance each other.
   - Examples:
     - Jaron Lanier integrates music and technology.
     - Reid Hoffman’s exposure to cognitive science greatly influen

In [33]:
print("=== TOOL TRACE ===")
for t in tool_trace_current(resp3):
    print(t)


=== TOOL TRACE ===
{'content_type': 'Content', 'name': 'graphrag_search', 'arguments': '{"query":"What recurring ideas or patterns appear across the documents?","search_method":"local"}', 'result': None}
{'content_type': 'Content', 'name': None, 'arguments': None, 'result': '{\n  "tool": "graphrag_search",\n  "input": {\n    "query": "What recurring ideas or patterns appear across the documents?",\n    "search_method": "local",\n    "params": null\n  },\n  "result": "### Recurring Ideas and Patterns in the Documents\\n\\nSeveral recurring ideas and themes emerge across the documents concerning technology, innovation, ethics, and interdisciplinary approaches. These interconnected ideas highlight notable overlaps in the thoughts and works of influential figures such as Jaron Lanier, Reid Hoffman, Judy Estrin, Kevin Scott, and key concepts like unsupervised learning. Below is a detailed exploration:\\n\\n---\\n\\n### Innovation as a Driving Force Across Disciplines\\n\\nOne prominent recu

In [35]:
add_section_title("Example 4: Routing to age_get_schema_cached", color="blue", size="18px")

<h2 style='color:blue; font-size:18px;'>Example 4: Routing to age_get_schema_cached</h2>

In [36]:
resp4 = await router_agent.run("Before querying, what does the graph look like?")

print(resp4)

The graph structure contains the following:

- **Vertex Labels:**
  - `"Document"`
  - `"Entity"`

- **Edge Labels:**
  - `"RELATED_TO"`

The graph appears to map relationships (edges labeled as `RELATED_TO`) between entities and/or documents.


In [37]:
print("=== TOOL TRACE ===")
for t in tool_trace_current(resp4):
    print(t)


=== TOOL TRACE ===
{'content_type': 'Content', 'name': 'age_get_schema_cached', 'arguments': '{"refresh":true}', 'result': None}
{'content_type': 'Content', 'name': None, 'arguments': None, 'result': '{\n  "tool": "age_get_schema_cached",\n  "refresh": true,\n  "cached": false,\n  "ttl_seconds": 3600,\n  "result": {\n    "graph": "graphRAG",\n    "vertex_labels": [\n      "\\"Document\\"",\n      "\\"Entity\\""\n    ],\n    "edge_labels": [\n      "\\"RELATED_TO\\""\n    ]\n  }\n}'}
